# Acervo que Fala — Notebook 04 (v8): as oito políticas da adjudicação

**Projeto final** · Inteligência Artificial Generativa & Large Language Models (ICA/PUC-Rio) · Eduardo Tosto

O lote v7 passou por dois crivos: a régua automática e uma **revisão de juiz** (Claude revisou os 17 cards com as fotos; o Eduardo adjudicou os 48 achados — concordância de ~95%). Da adjudicação saíram **oito políticas** que esta versão implementa:

1. **Foto sem resolução** → flag `falta_de_resolucao` e NENHUM texto — auditar o dataset é função assumida do projeto (o caso: uma foto de 100×66 px rendeu uma observação com detalhes que a imagem não tem como mostrar);
2. **Informação com flag não entra no texto** — a seção EM QUARENTENA do prompt lista o que as flags tiraram da redação (cores divergentes, medidas suspeitas, contradições);
3. **O catálogo manda nas palavras**: termo não-técnico do registro se usa como está (cabo, pá, algodão); jargão se traduz pelo glossário; descrição visual do catálogo ("figuras em X") tem força máxima;
4. **Espécie só com fonte**: "motivos zoomorfos" vira "figuras de animais" — nunca "onça" sem o catálogo dizer;
5. **Número literal do catálogo**: 41,5 não vira "cerca de 42";
6. **Alças e cordas fora** da medida e do foco;
7. **Função só quando acrescenta**: bolsa que guarda e panela que cozinha não se explicam;
8. **Contagem do catálogo prevalece**: "seis tubos" viaja pronto no prompt (`CONTAGENS DO REGISTRO`).

Duas mudanças de arquitetura acompanham: **validação com um retry** — o código confere o rascunho (marca de atribuição presente? alt ≤ 30 palavras? ausências? quarentena respeitada?) e, se falhou, devolve o diagnóstico ao modelo UMA vez para correção dirigida; e a **observação v3.1**, que pede todas as cores, inclusive as minoritárias (as penas azuis são a memória viva do porquê).

**Critério de saída desta etapa (decisão do Eduardo):** só se avança quando o resultado for satisfatório — zero reincidência dos defeitos adjudicados no gabarito e régua limpa nas checagens mecânicas.

*Metodologia: projeto construído com LLMs como suporte (vibe coding) — cada célula explicada.*

### Como rodar
1. **Ambiente de execução → Alterar o tipo → GPU T4** · 2. **Executar tudo** · 3. Tempo: **~45–55 min** (o retry adiciona alguns minutos). Deixe a aba aberta.

In [ ]:
# Etapa 1 — Instalação (Pillow travada, regra da casa) + checagem do ambiente
import PIL
!pip install -q -U transformers accelerate bitsandbytes sentence-transformers pillow=={PIL.__version__}
import torch, transformers
from PIL import ImageDraw
from torchvision.io import decode_image
print(f"transformers {transformers.__version__} | GPU: {torch.cuda.is_available()}")
print("ambiente íntegro ✓")

## Etapa 2 — Buscar os 20 objetos, com salvaguardas de imagem (~3 min)

O lote: os **5 objetos do smoke test** (para comparar com os notebooks anteriores) + **15 novos**, sorteados com seed fixa (reproduzível) entre os itens que **não** estão no conjunto de avaliação — o lote serve para testar o pipeline em escala, sem "viciar" nos itens que depois vão dar a nota.

Duas salvaguardas novas ao baixar cada foto:

- **Orientação EXIF**: fotos de câmera guardam a rotação numa etiqueta interna que os navegadores aplicam, mas o Python não — sem esta linha, o modelo poderia receber uma foto deitada sem ninguém saber. `ImageOps.exif_transpose` aplica a rotação correta.
- **Conversão para RGB**: garante que qualquer foto (escala de cinza, outros formatos de cor) chegue ao modelo no formato esperado.

Desta vez o registro completo de cada objeto (povo, materiais, dimensões, descrição curatorial...) **viaja junto** — a correção do bug do Notebook 03.

In [ ]:
import io, re, requests
from PIL import Image, ImageOps

BASE = "https://tainacan.museudoindio.gov.br/wp-json/tainacan/v2"
IDS_SMOKE = [9196, 665, 51023, 63283, 78838]
# 15 novos: sorteio seed 42, estratificado por categoria, excluindo os 50 casos
# de avaliação (seleção documentada no repositório, commit da E7)
IDS_LOTE = [1376, 84811, 883523, 2081, 5011, 200648, 210680, 5146, 500179, 3411, 1366, 4156, 205095, 905, 500322]

CAMPOS_REGISTRO = ["Nome do item", "Povo", "Categoria", "Matéria-prima",
                   "Técnica de confecção", "Dimensões", "Função",
                   "Estado de origem", "Ano de aquisição do objeto", "Descrição"]

objetos = []
for item_id in IDS_SMOKE + IDS_LOTE:
    item = requests.get(f"{BASE}/items/{item_id}", timeout=60).json()
    url_imagem = re.search(r'src="([^"]+)"', item["document_as_html"]).group(1)
    foto = Image.open(io.BytesIO(requests.get(url_imagem, timeout=90).content))
    foto = ImageOps.exif_transpose(foto).convert("RGB")  # salvaguardas
    meta_bruto = requests.get(f"{BASE}/item/{item_id}/metadata", timeout=60).json()
    todos = {m["metadatum"]["name"]: m["value_as_string"] for m in meta_bruto if m.get("value_as_string")}
    registro = {c: todos.get(c, "") for c in CAMPOS_REGISTRO}
    # Porteiro de resolução (5ª adjudicação): foto pequena demais não gera descrição
    # nenhuma — vira flag de dataset. O caso 5146 tinha 100×66 px e a observação
    # "descreveu" textura em espiral que a imagem não tem como mostrar.
    w, h = foto.size
    resolucao_ok = w * h >= 100_000
    objetos.append({"id": item_id, "titulo": item["title"], "foto": foto,
                    "registro": registro, "resolucao": f"{w}×{h}",
                    "resolucao_ok": resolucao_ok})
    alerta = "" if resolucao_ok else "  ⚠ SEM RESOLUÇÃO — não será descrito"
    print(f"✓ {item_id} — {item['title']} ({registro['Povo']}) {w}×{h}px{alerta}")
print(f"{len(objetos)} objetos carregados | "
      f"{sum(1 for o in objetos if not o['resolucao_ok'])} sem resolução mínima")

In [ ]:
# Etapa 3 — Rubrica v1.4 (ganhou 'gameliforme' no glossário) + RAG híbrido: categoria garantida + glossário recuperado
import json, os, requests
from google.colab import drive
from sentence_transformers import SentenceTransformer, util

drive.mount("/content/drive")
PROJETO = "/content/drive/MyDrive/00_IA/GenAI & LLMs - PUC/Projeto_LLM"
RUBRICA_DRIVE = f"{PROJETO}/dados/rubrica_v1_4.json"
RUBRICA_REPO = ("https://raw.githubusercontent.com/eduardotosto/acervo-que-fala/"
                "main/dados/rubrica/rubrica.json")
if os.path.exists(RUBRICA_DRIVE):
    with open(RUBRICA_DRIVE, encoding="utf-8") as f:
        rubrica = json.load(f)
    print(f"rubrica lida do Drive ✓")
else:
    rubrica = requests.get(RUBRICA_REPO, timeout=60).json()
    print("rubrica lida do repositório público (ainda não está no Drive) ✓")

trechos = rubrica["trechos"]
glossario = [t for t in trechos if t["categoria"] == "glossario"]
trechos_categoria = [t for t in trechos if t["categoria"] != "glossario"]
por_categoria = {}
for t in trechos_categoria:
    por_categoria.setdefault(t["categoria"], []).append(t)

# Embedder na CPU: 23 trechos e 20 consultas são trabalho trivial, e a T4 fica inteira
# para o modelo 4-bit (armadilha conhecida: device_map="auto" com a GPU já ocupada
# manda camadas para a CPU e quebra a geração).
embedder = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B", device="cpu")
vet_glossario = embedder.encode([t["texto"] for t in glossario], convert_to_tensor=True)
vet_categoria = embedder.encode([t["texto"] for t in trechos_categoria], convert_to_tensor=True)
# A família Qwen3-Embedding é instruída: a CONSULTA vai com o prompt de query, os
# documentos vão sem. Se a versão instalada não trouxer o prompt, seguimos sem ele.
PROMPT_QUERY = "query" if "query" in (getattr(embedder, "prompts", None) or {}) else None

def recuperar(categoria, consulta, k_glossario=2, k_fallback=2):
    """RAG híbrido. A diretriz da categoria vem do REGISTRO (o campo Categoria existe em
    100% dos itens) — é garantia, não sorteio; o glossário vem da busca semântica, que é
    onde a recuperação realmente agrega. Categoria fora da rubrica cai no modo semântico."""
    v = embedder.encode(consulta, prompt_name=PROMPT_QUERY, convert_to_tensor=True)
    scores = util.cos_sim(v, vet_glossario)[0]
    achados = [glossario[i] for i in scores.argsort(descending=True)[:k_glossario].tolist()]
    fixos = por_categoria.get(categoria)
    if fixos is None:
        s2 = util.cos_sim(v, vet_categoria)[0]
        fixos = [trechos_categoria[i] for i in s2.argsort(descending=True)[:k_fallback].tolist()]
    return fixos + achados

print(f"rubrica {rubrica['versao']}: {len(por_categoria)} categorias + {len(glossario)} "
      f"trechos de glossário | prompt de query: {PROMPT_QUERY or 'indisponível'} ✓")

In [ ]:
# Etapa 4 — Modelo (Qwen3-VL-8B em 4-bit, como nos notebooks anteriores)
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

MODELO = "Qwen/Qwen3-VL-8B-Instruct"
quantizacao = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
modelo = Qwen3VLForConditionalGeneration.from_pretrained(
    MODELO, quantization_config=quantizacao, device_map="auto"
)
processador = AutoProcessor.from_pretrained(MODELO)

def gerar(conteudo, max_tokens=400):
    conversa = [{"role": "user", "content": conteudo}]
    entradas = processador.apply_chat_template(
        conversa, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt"
    ).to(modelo.device)
    with torch.no_grad():
        saida = modelo.generate(**entradas, max_new_tokens=max_tokens)
    return processador.decode(saida[0][entradas["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def extrair_json(texto):
    texto = re.sub(r"^```(json)?|```$", "", texto.strip(), flags=re.MULTILINE).strip()
    inicio, fim = texto.find("{"), texto.rfind("}")
    return json.loads(texto[inicio:fim + 1])

print("modelo carregado ✓")

## Etapa 5 — Observação v3.1 e a varredura de artefatos (~15 min)

Uma linha nova no prompt da observação: **todas as cores, inclusive as minoritárias e as parcialmente encobertas** — mira os casos em que a observação não viu o preto e o azul que o registro nomeia (Pulseira 84811) e as penas azuis originais. O resto do prompt fica como no v7, para a comparação valer.

Itens **sem resolução mínima** (porteiro da Etapa 2) não passam por aqui: sem imagem confiável, não há observação — e a política da 5ª adjudicação é não gerar texto nenhum.

O parse continua tolerante às grafias do cabeçalho (o modelo escreveu FONDO em 19 de 20 observações no v6) e a flag de artefato vem da **varredura de todas as seções** — no v7 ela acertou 4/4 itens com 0 falso positivo, contra 2/4 da seção `ARTEFATOS` sozinha.

In [ ]:
# Etapa 5 — Observação v3.1 (uma linha nova: TODAS as cores — resto inalterado).
# O bloco de varredura de artefatos é o mesmo texto de avaliacao/checar_lote.py.
import collections

PROMPT_OBSERVACAO_V3 = """Você está diante da fotografia de um objeto do acervo de um museu — objetos etnográficos de povos indígenas do Brasil, fotografados em estúdio. Este contexto serve para você reconhecer materiais e situações de estúdio; NÃO use para adivinhar o que o objeto é ou significa. Descreva somente o que está visível NESTA fotografia.

Preencha as seções abaixo, nesta ordem:

OBJETO: o que se vê, em uma frase — forma geral, sem nomear função nem significado.
MATERIAIS E CORES: os materiais aparentes e suas cores, do maior para o menor — liste TODAS as cores presentes, inclusive as minoritárias e as parcialmente encobertas. Material que não dá para identificar recebe o termo genérico ("fibra", "madeira clara") — nunca chute espécie ou origem.
PADRÕES E TEXTURAS: desenhos, tramas e acabamentos visíveis, descritos pela forma (linhas, xadrez, diagonais) — somente se existirem.
PARTES E QUANTIDADES: partes distinguíveis e contáveis (tubos, furos, alças, penas destacadas).
POSIÇÃO: como o objeto está na foto (de pé, deitado, inclinado) e partes internas visíveis (boca, interior, verso).
LEGIBILIDADE: o que estiver ilegível ou incerto — declare a incerteza em vez de estimar.
FUNDO E ESTÚDIO: o fundo e qualquer artefato de estúdio (etiqueta, numeração, cartela de cores, régua, suporte).
ENQUADRAMENTO: inteiro OU detalhe — "detalhe" SÓ se a foto mostra claramente apenas parte do objeto; objeto que encosta ou sangra nas margens conta como inteiro.
ARTEFATOS: os artefatos de estúdio vistos, separados por vírgula, ou "nenhum".

Regra geral: o que não está visível não existe para esta descrição. Responda em português."""

for n, obj in enumerate(objetos, 1):
    if not obj["resolucao_ok"]:
        obj["observacao"] = ""
        print(f"[{n}/{len(objetos)}] {obj['titulo']} PULADO (sem resolução: {obj['resolucao']})")
        continue
    obj["observacao"] = gerar(
        [{"type": "image", "image": obj["foto"]}, {"type": "text", "text": PROMPT_OBSERVACAO_V3}],
        max_tokens=500,
    )
    print(f"[{n}/{len(objetos)}] {obj['titulo']} observado ✓")

# A secao ARTEFATOS sozinha nao basta. Medido no lote v6: em 2 dos 4 itens com artefato
# real, o modelo descreveu o artefato na secao onde o viu (PARTES E QUANTIDADES,
# LEGIBILIDADE, FUNDO E ESTUDIO) e respondeu "nenhum" em ARTEFATOS - ele nao repete o
# que ja disse. A varredura le todas as secoes e descarta as mencoes sob negacao
# ("nao ha etiquetas", "sem numeracao"), que sao a maioria. No lote v6: 4/4 itens
# corretos, 0 falso positivo, contra 2/4 da secao sozinha.
# "suporte" ficou de fora: nomeia tanto a base de estudio quanto uma parte do proprio
# objeto (a peca central da bracadeira 1366) - ambiguidade que nao da para resolver aqui.
FAMILIAS_ARTEFATO = [("etiqueta", r"etiqueta\w*|r[óo]tulo\w*"),
                     ("inscrição", r"numera[çc]\w*|marca[çc][ãa]o\w*|inscri[çc]\w*"),
                     ("cartela de cores", r"cartela\w*|escala\s+de\s+cores?"),
                     ("régua", r"r[ée]gua\w*")]
RE_TERMO_ARTEFATO = re.compile(
    r"(?<![a-zà-ú])(" + "|".join(p for _, p in FAMILIAS_ARTEFATO) + ")", re.I)
RE_NEGACAO = re.compile(r"(?<![a-zà-ú])(n[ãa]o\s|nenhum\w*|sem\s|nada\s|aus[êe]ncia)", re.I)


def artefatos_da_observacao(obs):
    """Uma flag por familia de artefato citada na observacao fora de contexto de
    negacao, com a frase mais curta como detalhe."""
    por_familia = {}
    for m in RE_TERMO_ARTEFATO.finditer(obs):
        ini = max(obs.rfind(".", 0, m.start()), obs.rfind("\n", 0, m.start())) + 1
        if RE_NEGACAO.search(obs[ini:m.start()]):
            continue
        fim = obs.find(".", m.end())
        frase = re.sub(r"\s+", " ", obs[ini:fim if fim > 0 else m.end() + 60]).strip(" .;,")
        frase = re.sub(r"^[#*\s]*[A-ZÀ-Ú ]{4,}[#*\s]*:[#*\s]*", "", frase)
        nome = next(n for n, pat in FAMILIAS_ARTEFATO if re.match(pat, m.group(0), re.I))
        if nome not in por_familia or len(frase) < len(por_familia[nome]):
            por_familia[nome] = frase
    return [{"tipo": "artefato_estudio", "detalhe": f"{n}: {t}"} for n, t in por_familia.items()]

# Cor vista na foto que o registro nao nomeia. Reencontra automaticamente o achado
# fundador do projeto: as duas penas AZUIS da Faixa Kalapalo (665), que o registro
# nao menciona. So dispara quando o registro DESCREVE cores (registro que nao fala de
# cor nao autoriza divergencia) e so para cores informativas - bege, marrom, cinza e
# afins sao a cor natural do material, que o catalogo nunca nomeia.
CORES_INFORMATIVAS = {"azul": r"azu[lm]|azulad", "verde": r"verde|esverdead",
                      "vermelho": r"vermelh|avermelhad", "amarelo": r"amarel",
                      "laranja": r"laranj", "roxo": r"rox|violet", "rosa": r"rosa",
                      "preto": r"pret", "branco": r"branc"}
CORES_DE_MATERIAL = r"bege|marrom|castanh|cinza|creme|ocre|dourad|prate|amarronzad"


def cores_do_texto(texto):
    return {nome for nome, pat in CORES_INFORMATIVAS.items()
            if re.search(rf"(?<![a-zà-ú])(?:{pat})\w*", texto or "", re.I)}


def cores_divergentes(observacao, registro, secao_fn):
    """Cores que a observacao nomeia e o registro nao. Devolve lista de flags."""
    texto_reg = " ".join(str(registro.get(c, "")) for c in
                         ("Descrição", "Matéria-prima", "Técnica de confecção", "Nome do item"))
    no_registro = cores_do_texto(texto_reg)
    if not no_registro:
        return []
    na_foto = cores_do_texto(secao_fn(observacao, "MATERIAIS E CORES") + " " +
                             secao_fn(observacao, "PADRÕES E TEXTURAS"))
    so_na_foto = sorted(na_foto - no_registro)
    if not so_na_foto:
        return []
    return [{"tipo": "divergencia_imagem_catalogo",
             "detalhe": f"a foto mostra {', '.join(so_na_foto)} e o registro nomeia só "
                        f"{', '.join(sorted(no_registro))}"}]

# --- Parse das seções ---
# O nome da seção volta com variações: no lote v6, 19 das 20 observações escreveram
# "FONDO E ESTÚDIO" e uma escreveu "FUNDOS E ESTÚDIO" — nenhuma acertou "FUNDO E
# ESTÚDIO". Com a grafia exigida exata, a seção não era lida NEM removida, e ninguém
# ficava sabendo. O padrão agora tolera a variação, e o relatório no fim da célula
# avisa quando um cabeçalho não aparece.
CABECALHO = (r"(?:OBJETO|MATERIAIS E CORES|PADRÕES E TEXTURAS|PARTES E QUANTIDADES|POSIÇÃO|"
             r"LEGIBILIDADE|F[OU]NDOS? E EST[ÚU]DIO|ENQUADRAMENTO|ARTEFATOS)")
CONSUMIDAS = r"(?:ENQUADRAMENTO|ARTEFATOS|F[OU]NDOS? E EST[ÚU]DIO)"
P = r"[#*]*\s*"

def secao(texto, nome):
    """Conteúdo de uma seção nomeada, ou string vazia. Aceita cabeçalho em negrito
    (**ENQUADRAMENTO:**) e em título markdown (### ENQUADRAMENTO:)."""
    m = re.search(rf"{nome}{P}:{P}(.+?)(?=\n{P}{CABECALHO}{P}:|\Z)", texto, re.S | re.I)
    return m.group(1).strip().strip("*#").strip() if m else ""

faltando = collections.Counter()
for obj in objetos:
    obs = obj["observacao"]
    if not obs:  # sem resolução: campos vazios, flag entra na redação
        obj.update(enquadramento="", enquadramento_ok=True, artefatos_secao=[],
                   flags_artefato=[], artefatos_obs=[], observacao_para_redacao="",
                   consulta_rag=obj["titulo"], marca_atribuicao="", flags_cor=[])
        continue
    enq = secao(obs, "ENQUADRAMENTO").lower()
    obj["enquadramento"] = "detalhe" if enq.startswith("detalhe") else "inteiro"
    obj["enquadramento_ok"] = enq.startswith(("inteiro", "detalhe"))
    # o que a seção ARTEFATOS respondeu, guardado só para medir a diferença
    art = secao(obs, "ARTEFATOS")
    obj["artefatos_secao"] = [] if (not art or art.lower().startswith("nenhum")) else [
        a.strip(" .") for a in art.split(",") if a.strip(" .")]
    # a flag vem da VARREDURA de todas as seções, não da obediência a uma delas
    obj["flags_artefato"] = artefatos_da_observacao(obs)
    obj["artefatos_obs"] = [f["detalhe"] for f in obj["flags_artefato"]]
    for nome in ("OBJETO", "MATERIAIS E CORES", "LEGIBILIDADE", "F[OU]NDOS? E EST[ÚU]DIO",
                 "ENQUADRAMENTO", "ARTEFATOS"):
        if not secao(obs, nome):
            faltando[nome] += 1
    # A redação recebe a observação sem as três seções que o código já consumiu.
    obj["observacao_para_redacao"] = re.sub(
        rf"\n?{P}{CONSUMIDAS}{P}:.*?(?=\n{P}{CABECALHO}{P}:|\Z)", "", obs, flags=re.S | re.I).strip()
    obj["consulta_rag"] = (f"{obj['titulo']}. {secao(obs, 'OBJETO')} "
                           f"{secao(obs, 'PADRÕES E TEXTURAS')}")[:400]

ok = sum(1 for o in objetos if o["enquadramento_ok"])
so_varredura = sum(1 for o in objetos if o["flags_artefato"] and not o["artefatos_secao"])
vazou = sum(1 for o in objetos if re.search(r"(?<![a-zà-ú])fundo\b", o["observacao_para_redacao"], re.I))
print(f"\nparse: {ok}/{len(objetos)} com ENQUADRAMENTO válido | "
      f"{sum(1 for o in objetos if o['enquadramento'] == 'detalhe')} 'detalhe'")
print(f"artefatos: {sum(len(o['flags_artefato']) for o in objetos)} flags pela varredura, "
      f"{sum(len(o['artefatos_secao']) for o in objetos)} pela seção ARTEFATOS "
      f"({so_varredura} itens que só a varredura pegou)")
print(f"seções ausentes por cabeçalho: {dict(faltando) or 'nenhuma'} | "
      f"{vazou} observações ainda citam 'fundo' depois da limpeza")

# A fórmula de atribuição vem sorteada pelo CÓDIGO, não pedida ao modelo. No lote v6,
# "segundo o registro do museu" abriu 20 de 20 textos: é o primeiro exemplo do prompt,
# copiado literalmente — o mecanismo do papagaio de exemplo. Pedir "formulação variada"
# não funciona: o modelo não sabe o que escreveu no item anterior. O código sabe.
FORMULAS_ATRIBUICAO = [
    "segundo o registro do museu",
    "o registro do museu informa que",
    "conforme o catálogo do museu",
    "de acordo com a ficha do museu",
    "o catálogo do museu registra que",
]
for n, obj in enumerate(objetos):
    obj["marca_atribuicao"] = FORMULAS_ATRIBUICAO[n % len(FORMULAS_ATRIBUICAO)]
    # cor vista na foto que o registro não nomeia — recupera sozinha o achado fundador
    # do projeto (as duas penas azuis da Faixa Kalapalo, ausentes do registro)
    obj["flags_cor"] = cores_divergentes(obj["observacao"], obj["registro"], secao)

print(f"cores: {sum(1 for o in objetos if o['flags_cor'])} itens com cor que o registro não nomeia "
      f"| atribuição: {len(FORMULAS_ATRIBUICAO)} fórmulas em rodízio")

## Etapa 5b — o que o código extrai do registro (~4 min)

Aritmética e comparação não se pedem a modelo. Este bloco (o mesmo texto de `avaliacao/checar_lote.py`) entrega prontos ao prompt:

- **ESCALA** — a maior dimensão, com o número LITERAL do catálogo (41,5 não vira 42), ignorando medidas com cordel ou alça (política 6). Se qualquer flag de dimensão disparou (teto de plausibilidade, dimensão atipicamente pequena, alças soltas), a escala vira instrução de **quarentena**: nenhuma medida no texto;
- **CONTAGENS DO REGISTRO** — "seis tubos" extraído da Descrição; prevalece sobre a contagem visual (política 8; o modelo contou sete tubos em quatro lotes seguidos);
- **flags de metadado** — teto e piso de plausibilidade por categoria (calibrados no próprio acervo), ano impossível, alças soltas, e a contradição **miniatura×função de uso real** agora detectada em código (a pergunta isolada ao modelo falhou duas vezes no caso brinquedo×caça — heurística determinística resolve o padrão conhecido; a pergunta continua para o resto).

In [ ]:
# Etapa 5b — o que o CÓDIGO extrai do registro (aritmética não se pede a modelo).
# Este bloco é o mesmo texto de avaliacao/checar_lote.py, para que o notebook e a
# ferramenta que mede os lotes antigos nunca divirjam. Nesta versão ele carrega as
# decisões da 5ª adjudicação: número LITERAL do catálogo (41,5 não vira 42), medida com
# cordel/alça fora da escala, e QUARENTENA — flag sobre a dimensão tira a medida do
# texto; miniatura×função de uso real vira flag de contradição sem depender do modelo.
import collections

ROTULOS = r"(comprimento|altura|largura|di[âa]metro|espessura|profundidade)"
RE_ROTULO, RE_NUM = re.compile(ROTULOS, re.I), re.compile(r"\d+(?:[.,]\d+)?")
# medida "com cordel / com a alça esticada" mede o objeto pendurado, não a peça
RE_COM_CORDA = re.compile(
    r"com\s+(a\s+|o\s+)?(cordel|cord[aã]o|al[çc]a|amarra)|esticad|\+\s*(cordel|cord[aã]o|al[çc]a)", re.I)

# Teto de plausibilidade por categoria = Q3 + 3xIQR das dimensões do PRÓPRIO acervo
# (547 dos 555 itens de dados/itens.json têm medida parseável), com piso de 150 cm —
# abaixo disso, peça grande é plausível em qualquer categoria. No acervo inteiro o teto
# dispara 2 vezes: o abano de 290 cm (dimensão improvável, caso conhecido do projeto) e
# a capa de pele de onça de 223 cm (peça genuinamente grande). Flag é pedido de
# conferência humana, não veredito — a taxa de alarme é o que importa, e é de 0,4%.
TETOS_CATEGORIA = {
    "Adornos Plumários": 156,
    "Adornos de Materiais Ecléticos, Indumentária e Toucador": 106,
    "Armas": 491,
    "Cerâmica": 56,
    "Cordões e Tecidos": 103,
    "Etnobotânica": 33,
    "Instrumentos musicais e de sinalização": 76,
    "Objetos rituais, mágicos e lúdicos": 144,
    "Trançados": 181,
    "Utensílios e implementos de materiais ecléticos": 75,
}
PISO_SUSPEITA = 150
# miniatura: peça bem menor que o comum da categoria, em categorias de objeto grande
MEDIANAS_CATEGORIA = {"Cerâmica": 14.0, "Trançados": 41.5, "Armas": 212.5,
                      "Instrumentos musicais e de sinalização": 41.0,
                      "Utensílios e implementos de materiais ecléticos": 19.5}

def escala_do_registro(dimensoes):
    """Maior dimensão do objeto, em cm, com o rótulo — a regra editorial 25 ("escala é a
    maior dimensão, nunca a medida de uma parte"), resolvida em código. Trata a lista
    enumerada ("29,5; 21,5; ... e 9,5 cm de comprimento") herdando o rótulo do segmento
    seguinte. Devolve (valor, rótulo) ou None."""
    if not dimensoes or "cm" not in dimensoes.lower():
        return None
    segmentos = [s for s in re.split(r";|\s-\s", dimensoes) if s.strip()]
    candidatos = []
    for i, seg in enumerate(segmentos):
        if RE_COM_CORDA.search(seg):
            continue
        rot = RE_ROTULO.search(seg) or next(
            (RE_ROTULO.search(s) for s in segmentos[i + 1:] if RE_ROTULO.search(s)), None)
        rotulo = rot.group(1).lower() if rot else ""
        candidatos += [(float(m.group(0).replace(",", ".")), rotulo) for m in RE_NUM.finditer(seg)]
    return max(candidatos) if candidatos else None


def numero_pt(v):
    """O número exatamente como o catálogo o dá, em grafia brasileira — arredondar cria
    um número que não existe no registro (5ª adjudicação: 'cerca de 42' para 41,5)."""
    return f"{v:g}".replace(".", ",")


def analisar_registro(registro):
    """Devolve a linha ESCALA pronta para o prompt + as flags de metadado que a
    aritmética já resolve (dimensão fora do teto da categoria, ano impossível)."""
    cat, flags = registro.get("Categoria", ""), []
    e = escala_do_registro(registro.get("Dimensões", ""))
    if not e:
        escala = "não informada no registro — não escreva medida nenhuma"
    else:
        valor, rotulo = e
        escala = f"cerca de {numero_pt(valor)} cm" + (f" de {rotulo}" if rotulo else "")
        mediana = MEDIANAS_CATEGORIA.get(cat)
        if mediana and valor < 10 and valor < mediana / 2:
            # 4a revisao: dimensao atipicamente pequena pede conferencia humana, pelo
            # mesmo motivo do abano de 290 cm — miniatura genuina ou erro de registro
            flags.append({"tipo": "metadado_suspeito", "detalhe":
                          f"dimensão atipicamente pequena para {cat} ({numero_pt(valor)} cm; "
                          f"mediana da categoria {numero_pt(mediana)} cm) — miniatura genuína "
                          f"ou erro de registro; conferir"})
        teto = TETOS_CATEGORIA.get(cat)
        if teto and valor > max(teto, PISO_SUSPEITA):
            flags.append({"tipo": "metadado_suspeito", "detalhe":
                          f"{numero_pt(valor)} cm de {rotulo or 'dimensão'} está acima do teto de "
                          f"plausibilidade da categoria {cat} ({teto} cm, calculado do acervo)"})
    if re.search(r"al[çc]as? soltas?", registro.get("Descrição", "") or "", re.I) and e:
        flags.append({"tipo": "metadado_suspeito", "detalhe":
                      "a Descrição menciona alças soltas — o comprimento pode incluí-las; conferir"})
    desc = registro.get("Descrição", "") or ""
    if re.search(r"(?<![a-zà-ú])miniatura", desc, re.I) and re.search(
            r"ca[çc]a|pesca|ataque|guerra|defesa", registro.get("Função", "") or "", re.I):
        flags.append({"tipo": "metadado_suspeito", "detalhe":
                      "a Descrição diz miniatura/brinquedo e a Função descreve uso real "
                      "(caça, ataque...) — contradição entre campos do registro; conferir"})
    ano = (registro.get("Ano de aquisição do objeto") or "").strip()
    if ano.isdigit() and not (1850 <= int(ano) <= 2026):
        flags.append({"tipo": "metadado_suspeito", "detalhe": f"ano de aquisição improvável: {ano}"})
    # 5a adjudicacao: informacao flagada nao entra no texto. Qualquer flag sobre a
    # dimensao (teto, piso, alcas soltas) poe a medida em quarentena.
    if any(re.search(r"cm|alças soltas|dimensão", f["detalhe"]) for f in flags):
        escala = ("EM QUARENTENA — a medida do registro está sob conferência (flag): "
                  "não escreva medida nenhuma")
    return escala, flags

NUMEROS_PT = {"um": 1, "uma": 1, "dois": 2, "duas": 2, "três": 3, "tres": 3,
              "quatro": 4, "cinco": 5, "seis": 6, "sete": 7, "oito": 8, "nove": 9,
              "dez": 10, "onze": 11, "doze": 12}
RE_NUM_SUBST = re.compile(r"(?<![a-zà-ú])(" + "|".join(NUMEROS_PT) + r"|[2-9]|1[0-2])\s+([a-zà-ú]{3,}s)(?![a-zà-ú])", re.I)


def contagens(texto):
    """Pares (substantivo plural, número) escritos num texto: 'seis tubos' -> (tubos, 6)."""
    pares = {}
    for m in RE_NUM_SUBST.finditer(texto or ""):
        n = NUMEROS_PT.get(m.group(1).lower()) or int(m.group(1))
        pares.setdefault(m.group(2).lower(), n)
    return pares

# Contradição entre campos do registro: pergunta ISOLADA, fora da tarefa de escrita.
# (A heurística miniatura×função acima já cobre o caso conhecido em código; a pergunta
# continua para contradições que a heurística não enxerga.)
PROMPT_CONTRADICAO = """Leia os campos do registro de catálogo de um museu abaixo e responda se existe CONTRADIÇÃO INTERNA: dois campos que afirmam coisas incompatíveis sobre o MESMO objeto.

Exemplo de contradição: o campo Descrição diz "brinquedo em miniatura" e o campo Função diz "utilizado para caça" — o mesmo objeto não é as duas coisas.
Não é contradição: campo vazio, informação que falta, detalhe que só um campo traz, ou estilo de escrita.

REGISTRO:
{registro}

Responda APENAS com JSON: {{"contradicao": true ou false, "campos": ["campo A", "campo B"], "detalhe": "uma frase explicando"}}"""

for n, obj in enumerate(objetos, 1):
    registro_txt = "\n".join(f"{k}: {v}" for k, v in obj["registro"].items() if v)
    obj["escala"], obj["flags_registro"] = analisar_registro(obj["registro"])
    # contagens do catálogo ("seis tubos") viajam prontas — prevalecem sobre a visual
    pares = contagens(obj["registro"].get("Descrição", ""))
    obj["contagem_registro"] = ("; ".join(f"{v} {k}" for k, v in pares.items())
                                or "nenhuma no registro")
    ja_flagado = any("miniatura" in f["detalhe"] for f in obj["flags_registro"])
    try:
        r = extrair_json(gerar([{"type": "text", "text":
                                 PROMPT_CONTRADICAO.format(registro=registro_txt)}], max_tokens=250))
        obj["contradicao"] = r
        if r.get("contradicao") and r.get("detalhe") and not ja_flagado:
            obj["flags_registro"].append({"tipo": "metadado_suspeito", "detalhe":
                                          f"contradição no registro: {r['detalhe']}"})
    except Exception:
        obj["contradicao"] = None
    marca = f" | {len(obj['flags_registro'])} flag(s)" if obj["flags_registro"] else ""
    print(f"[{n}/{len(objetos)}] {obj['id']} escala: {obj['escala'][:52]}{marca}")

print()
print(f"{sum(len(o['flags_registro']) for o in objetos)} flags de metadado geradas pelo código; "
      f"{sum(1 for o in objetos if 'QUARENTENA' in o['escala'])} medidas em quarentena")

## Etapa 6 — Redação v12: quarentena, catálogo-manda e retry (~25 min)

O prompt ganhou três estruturas novas:

- **EM QUARENTENA** — a lista do que as flags tiraram do texto, montada pelo código (política 2: flag = quarentena; o texto só carrega o que passou limpo);
- **O CATÁLOGO MANDA NAS PALAVRAS** (cláusula 2 do contrato) — a hierarquia de vocabulário decidida na adjudicação: termo não-técnico do registro entra como está; jargão sai pelo glossário; espécie nunca sem fonte;
- **CONTAGENS DO REGISTRO** injetadas — quantidade não é mais decisão do redator.

E a garantia nova de arquitetura: **validação com um retry**. O código confere o rascunho contra as exigências que o 8B mais perde (marca de atribuição presente no texto — 13/20 a omitiram no v7 mesmo recebendo a fórmula pronta; alt ≤ 30 palavras — 10/20 estouraram; frases de ausência; quarentena respeitada) e, quando falha, devolve o rascunho com o diagnóstico para UMA correção dirigida. É o padrão validador-e-reescrita: barato, limitado, e auditável — o campo `retry` fica salvo no resultado.

In [ ]:
PROMPT_REDACAO_V12 = """Você escreve descrições de acessibilidade para o acervo digital de um museu. Elas serão OUVIDAS por pessoas cegas, através de leitores de tela — escreva em linguagem cotidiana, com frases que funcionam no ouvido (ordem direta, sem parênteses longos).

INSUMOS — cada um autoriza um tipo de informação:

OBSERVAÇÃO VISUAL DA FOTOGRAFIA (autoriza: aparência — o que é visível):
{observacao}

ENQUADRAMENTO DECIDIDO NA OBSERVAÇÃO: {enquadramento}

ESCALA CALCULADA DO REGISTRO: {escala}

CONTAGENS DO REGISTRO (prevalecem SEMPRE sobre a contagem visual): {contagem}

MARCA DE ATRIBUIÇÃO DESTE TEXTO (use esta fórmula, exatamente): {marca}

EM QUARENTENA — informação flagada para revisão humana. NÃO escreva nada sobre:
{quarentena}

REGISTRO DO MUSEU (autoriza: fatos — sempre com atribuição; o título nomeia o objeto):
{registro}

DIRETRIZES PARA ESTE TIPO DE OBJETO (autorizam: vocabulário e o que observar nesta categoria):
{diretrizes}

CONTRATO DE FONTES — prevalece sobre qualquer outra regra:
1. Cada informação escrita precisa de fonte: visível na observação, escrita no registro (e então leva atribuição) ou vocabulário das diretrizes. Sem fonte, não escreva — omitir é sempre permitido; um texto curto e todo verificável vale mais que um completo com um palpite.
2. O CATÁLOGO MANDA NAS PALAVRAS: termo do registro que não é técnico se usa como está (cabo, pá, algodão, "dobradura afunilada") — não invente sinônimo nem nova forma de dizer o que o catálogo já diz bem; descrição visual do registro em linguagem comum (ex.: figuras em "X") entra com as palavras dele. Termo técnico do registro se traduz pelo glossário das diretrizes ("motivos zoomorfos" vira "figuras de animais"; "borda extrovertida" vira "boca que se abre para fora"); espécie de animal ou planta SÓ se o registro nomear — nunca pela aparência.
3. Incerteza se herda: o que estiver na seção LEGIBILIDADE da observação, ou vier com hesitação ("parece", "talvez", "possivelmente"), sai do texto ou vira o termo genérico — nunca vira afirmação.
4. Divergência não se resolve no texto: quando observação e registro conflitam, não escolha um lado nem harmonize — registre em flags como divergencia_imagem_catalogo e NÃO escreva a informação divergente. Quantidade: use SEMPRE o número de CONTAGENS DO REGISTRO; cores: use as que o REGISTRO nomeia (cor vista que o registro não nomeia já está em quarentena via flag).
5. Os exemplos deste prompt mostram a FORMA das frases, com lacunas [assim]; preencha sempre com o conteúdo deste objeto, nunca com as palavras do exemplo.

PRODUZA TRÊS SAÍDAS:

A) alt_text — o que a fotografia mostra, para quem não a vê.
   Uma frase, no máximo 30 palavras — o limite de 30 VENCE qualquer outra exigência desta seção: se faltar espaço, corte adjetivo e detalhe secundário, nunca o objeto e o povo.
   Começa pelo objeto (nomeado pelo TÍTULO do registro) e pelo povo, com o nome do povo em maiúscula inicial.
   Contém: o material; as cores que o registro nomeia, onde a cor informa; a forma dos padrões, sempre pela geometria OU pelo termo que a diretriz/glossário nomear (espinha-de-peixe, gregas) — quando a diretriz nomeia o padrão que você vê, USE esse termo; NUNCA descreva padrão por semelhança com letra, figura ou objeto ("em forma de G", "como uma flor", "lembra uma coroa" são erros) — exceto quando o próprio registro usar a comparação (figuras em "X").
   Quando a peça tem pintura ou decoração aplicada sobre a base, primeiro a decoração e suas cores, depois a base — "pintura [tipo] em [cores] sobre [material da base]".
   Se o ENQUADRAMENTO diz "detalhe", comece com "Detalhe de [objeto]"; se diz "inteiro", não mencione enquadramento nem orientação.
   Aves de penas: só as que o registro nomear, com as cores QUE O REGISTRO dá a elas: "penas [cores do registro] de [ave]".
   Não aparecem aqui: fundo ou estúdio, artefato de inventário, medida, alça/cordel/corda (salvo se for a característica principal do objeto), frases de ausência ("sem X", "não há X") e palavras de seção da observação ("partes contáveis", "legível").

B) descricao_objeto — o objeto em si, para quem quer conhecê-lo além da foto. Dois parágrafos, no máximo 120 palavras somadas.
   1º: a PRIMEIRA PALAVRA é o nome do objeto — sem "O objeto é…", "Trata-se de…", "é um objeto". A função entra nessa primeira frase SÓ quando diz algo que o nome não diz (caça, ritual, preparo) — bolsa que guarda, panela que cozinha, remo que rema e pulseira no pulso NÃO se explicam. Depois, a aparência: formas, materiais e padrões, cada informação uma vez.
   2º: os fatos do catálogo, AGRUPADOS, em frases naturais ("adquirido em [ano]"): a marca de atribuição aparece UMA vez, ao entrar neles, com a fórmula exata de MARCA DE ATRIBUIÇÃO — o texto PRECISA conter essa fórmula. A escala é EXATAMENTE a de ESCALA CALCULADA DO REGISTRO, em frase natural — não recalcule, não escolha outra medida, não cite medida de parte nem de alça; se lá diz para não escrever medida, NÃO escreva.
   Este texto descreve o objeto, não a fotografia: posição, fundo, enquadramento, "visível/parcialmente visível" não existem aqui; relações de ponto de vista viram relações da peça ("decrescentes", "em degraus").
   O texto diz o que existe: nada de "sem [coisa]", "não há [coisa]"; o que o registro não traz não vira frase.

C) flags — o que precisa de revisão humana. Artefatos, metadados improváveis e cores divergentes já foram registrados pelo código; concentre-se no que só quem leu os dois lados percebe:
   - divergencia_imagem_catalogo: conflito entre observação e registro (quantidade, material, formato) — ou objeto visto diferente do que o título nomeia (use o título no texto e registre aqui).
   Lista vazia [] se não houver nada.

Responda APENAS com JSON: {{"alt_text": "...", "descricao_objeto": "...", "flags": [{{"tipo": "...", "detalhe": "..."}}]}}"""

# Pós-processamento conservador: remove o fundo de estúdio residual SÓ quando o trecho
# termina em pontuação dentro de no máximo duas palavras ("..., sobre fundo bege." ✓).
# Em "sobre fundo bege e boca larga" ele não casa e não remove nada — amputar a frase
# seria pior que deixar passar, e o que sobra a verificação acusa.
RE_FUNDO = re.compile(
    r",?\s*\b(?:sobre|em|contra|com|num|no|sob)\s+(?:um |uma |o |a )?fundo\b"
    r"(?:\s+[a-zà-ú-]+){0,2}\s*(?=[,.;]|$)", re.I)

def pos_processar_alt(alt):
    alt = RE_FUNDO.sub("", alt)
    alt = re.sub(r",\s*,", ",", alt)
    alt = re.sub(r"\s{2,}", " ", alt).strip(" ,")
    if alt and not alt.endswith("."):
        alt += "."
    return alt

# ---- Quarentena: a lista do que as flags tiraram do texto (5ª adjudicação) ----
def montar_quarentena(obj):
    itens = []
    for f in obj["flags_cor"]:
        m = re.search(r"a foto mostra ([^e]+?) e o registro", f["detalhe"])
        if m:
            itens.append(f"as cores {m.group(1).strip()} (vistas na foto, ausentes do registro)")
    for f in obj["flags_registro"]:
        det = f["detalhe"]
        if "miniatura/brinquedo" in det or "contradição" in det:
            itens.append("a função do registro e a palavra miniatura (contradição flagada)")
        elif re.search(r"cm|dimensão|alças", det):
            itens.append("qualquer medida ou tamanho")
    if not obj["resolucao_ok"]:
        itens.append("TUDO — sem resolução de imagem, não há descrição")
    return chr(10).join(f"- {x}" for x in itens) if itens else "- nada em quarentena"

# ---- Validação + 1 retry: as exigências que o 8B costuma perder de primeira ----
def validar_rascunho(obj, alt, descricao):
    erros = []
    if len(alt.split()) > 30:
        erros.append(f"o alt tem {len(alt.split())} palavras — corte para no máximo 30")
    if obj["marca_atribuicao"] and obj["marca_atribuicao"] not in descricao.lower():
        erros.append(f"a descrição precisa conter a fórmula exata: {obj['marca_atribuicao']}")
    for texto, nome in ((alt, "alt_text"), (descricao, "descricao_objeto")):
        m = re.search(r"(?<![a-zà-ú])(sem [a-zà-ú]+|não há)", texto, re.I)
        if m:
            erros.append(f"remova a frase de ausência ('{m.group(0)}') do {nome}")
    if "QUARENTENA" in obj["escala"] and re.search(r"\d+[.,]?\d*\s*cm", descricao):
        erros.append("a medida está em quarentena — remova toda medida da descrição")
    return erros

for n, obj in enumerate(objetos, 1):
    if not obj["resolucao_ok"]:
        obj.update(alt_bruto="", alt_text="", descricao_objeto="", diretrizes_usadas=[],
                   retry=False, json_valido=True,
                   flags=[{"tipo": "falta_de_resolucao",
                           "detalhe": f"imagem de {obj['resolucao']} px — sem resolução para "
                                      f"descrever; item devolvido ao dataset"}] + obj["flags_registro"])
        print(f"[{n}/{len(objetos)}] {obj['titulo']}: SEM RESOLUÇÃO — flag, nenhum texto")
        continue
    registro_txt = chr(10).join(f"{k}: {v}" for k, v in obj["registro"].items() if v)
    achados = recuperar(obj["registro"]["Categoria"], obj["consulta_rag"])
    obj["diretrizes_usadas"] = [t["id"] for t in achados]
    prompt = PROMPT_REDACAO_V12.format(
        observacao=obj["observacao_para_redacao"], enquadramento=obj["enquadramento"],
        escala=obj["escala"], contagem=obj["contagem_registro"],
        marca=obj["marca_atribuicao"], quarentena=montar_quarentena(obj),
        registro=registro_txt, diretrizes=chr(10).join(f"- {t['texto']}" for t in achados),
    )
    obj["retry"] = False
    try:
        saida = extrair_json(gerar([{"type": "text", "text": prompt}], max_tokens=700))
        erros = validar_rascunho(obj, saida["alt_text"], saida["descricao_objeto"])
        if erros:
            # 1 retry com o diagnóstico — validador-e-reescrita, limitado a uma rodada
            obj["retry"] = True
            correcao = (prompt + chr(10) + chr(10) + "SEU RASCUNHO ANTERIOR:" + chr(10)
                        + json.dumps(saida, ensure_ascii=False) + chr(10) + chr(10)
                        + "CORRIJA APENAS ISTO e devolva o MESMO JSON completo:" + chr(10)
                        + chr(10).join(f"- {e}" for e in erros))
            saida = extrair_json(gerar([{"type": "text", "text": correcao}], max_tokens=700))
        obj["alt_bruto"] = saida["alt_text"]
        obj["alt_text"] = pos_processar_alt(saida["alt_text"])
        obj["descricao_objeto"] = saida["descricao_objeto"]
        flags_modelo = [f for f in saida.get("flags", [])
                        if f.get("tipo") not in ("artefato_estudio", "metadado_suspeito")]
        obj["flags"] = (obj["flags_artefato"] + obj["flags_registro"] + obj["flags_cor"]
                        + flags_modelo)
        obj["json_valido"] = True
    except Exception:
        obj["alt_bruto"] = obj["alt_text"] = ""
        obj["descricao_objeto"] = ""
        obj["flags"] = obj["flags_registro"] + obj["flags_cor"]
        obj["json_valido"] = False
    marca_retry = " (retry)" if obj["retry"] else ""
    print(f"[{n}/{len(objetos)}] {obj['titulo']}{marca_retry}: {obj['alt_text'][:70]}...")

print(f"{chr(10)}retries: {sum(1 for o in objetos if o.get('retry'))}/20 | "
      f"pós-processamento de fundo agiu em "
      f"{sum(1 for o in objetos if o.get('alt_bruto') != o['alt_text'])} alts")

## Etapa 7 — Verificação automática

As checagens usam **fronteira de palavra**. A versão anterior procurava o termo como pedaço de texto: "parece" casava com "aparece", "fundo" com "profundo", "coração" com "decoração" — o mesmo casamento raso que o projeto diagnosticou nos modelos estava na própria régua que os media.

Checagens: JSON válido; povo no alt; artefato nos textos; até 30 palavras no alt; atribuição ao registro; frases-etiqueta; "foi aquisição em"; afirmações de ausência; foto e jargão de fotografia; especulação; frases vazias; "fundo" no alt; qualidade das flags; coerência de enquadramento; medidas que não existem no registro; escala pela medida errada; miniatura declarada; teto de escuta do nível 2.

Nova aqui: **artefato visto sem flag** — a varredura acha o artefato na observação e nenhuma flag foi gerada. É a checagem que teria mostrado, no lote v6, os dois artefatos que a seção `ARTEFATOS` deixou passar.

A função pula sozinha as checagens cujo campo não existe no item, e é a mesma de `avaliacao/checar_lote.py` — que aplica esta régua aos lotes anteriores, para que v5, v6, v7 e o Gemma sejam comparáveis.

In [ ]:
# Etapa 7 — Verificação automática. Este bloco é o mesmo texto de
# avaliacao/checar_lote.py, que aplica estas checagens aos lotes anteriores — é assim
# que o v8 e os lotes antigos são comparados pela mesma régua.
# Todos os termos são procurados com FRONTEIRA DE PALAVRA. A versão anterior usava
# "termo in texto": "parece" casava com "aparece", "fundo" com "profundo", "coração"
# com "decoração" — o mesmo casamento raso de texto que o projeto diagnosticou nos
# modelos aparecia na própria régua que os media.
B = lambda termos: re.compile(r"(?<![a-zà-úA-ZÀ-Ú])(?:" + "|".join(termos) + r")", re.I)
RE_ARTEFATO = B(["cartela", "paleta", "numeraç", "marcaç", "etiqueta", "régua", "suporte",
                 "rótulo", "rotulo", "inscriç", "tombo"])
RE_AUSENCIA = re.compile(
    r"(?<![a-zà-ú])(?:n[ãa]o (?:h[áa]|é|s[ãa]o|est[áa]|apresenta|possui|tem|cont[ée]m|traz|"
    r"exibe|permite|menciona|descrev\w+|inform\w+|registr\w+|visív\w+)|sem \w+|nenhum\w*|"
    r"aus[êe]ncia de)(?!\s*\w{0,12}identificad)", re.I)
RE_ESPECULACAO = B(["sugere", "sugerindo", "parece", "parecendo", "possivelmente", "talvez"])
RE_VAZIA = B(["porte médio", "uso prático", "uso frequente", "sinais de uso", "forma funcional",
              "forma é funcional"])
RE_FUNDO_TXT = B(["fundo"])
RE_FOTO = re.compile(r"(?<![a-zà-ú])(?:posicionad|enquadr|fotografia|[dn]a imagem|ao fundo|"
                     r"plano (?:médio|geral|fechado|aberto)|close|"
                     r"inclinad\w*\s+(?:levemente\s+)?(?:para\s+)?[aà]?\s*"
                     r"(?:direita|esquerda|frente|trás))", re.I)
# jargão de fotografia no alt: "close-up" e "plano médio" apareceram como
# vocabulário novo de enquadramento na v5, depois que a regra proibiu
# "inteiro/horizontal/vertical". "Detalhe de..." continua sendo a marca sancionada.
RE_JARGAO_FOTO = re.compile(r"(?<![a-zà-ú])(?:close|plano (?:médio|geral|fechado|aberto)|primeiro plano)", re.I)
# funcao tautologica: a que so repete o que o nome do objeto ja diz (regra 21)
RE_FUNCAO_OBVIA = re.compile(
    r"(pulseira|bracelete)[^.]{0,45}(pulso|braço)|(flauta|instrumento)[^.]{0,50}(som|sonor|músic|music)"
    r"|(bolsa|cesto|cesta)[^.]{0,45}(guardar|transportar|carregar)|(pote|panela|vasilha|tigela)"
    r"[^.]{0,50}(armazenar|guardar|conter)|(remo)[^.]{0,35}(remar|navega)|(arco)[^.]{0,35}(atirar|flecha)"
    r"|(colar|cinto|tanga)[^.]{0,40}(pescoço|cintura|corpo)"
    r"|(remo)[^.]{0,50}(desloc|vias aquáticas|transporte)|(panela)[^.]{0,50}(cozinhar|servir)", re.I)
# padrao descrito por semelhanca em vez de geometria: as gregas do 84811 viraram
# "elementos em forma de G ou C invertidos" com o termo certo disponivel no glossario
RE_ANALOGIA = re.compile(
    r"(?<![a-zà-ú])(?:em forma de\s+[\"“‘']?[A-Z][\"”’']?(?![a-zà-ú])|letra\s+[A-Z]\b"
    r"|(?:em )?forma de (?:flor|coração|estrela|coroa|esteira|pétala|folha|animal|ave|leque)"
    r"|lembra(?:ndo)? um|semelhante a um|parecid\w+ com)", re.I)
# marca de atribuicao: uma por bloco de fatos do catalogo, nao uma por fato
RE_MARCA_ATRIB = re.compile(
    r"segundo o registro(?: do museu)?|o registro(?: do museu)? (?:informa|menciona|indica|descreve|diz)"
    r"|conforme o (?:registro|cat[áa]logo)|de acordo com o (?:registro|cat[áa]logo)"
    r"|segundo o cat[áa]logo|o cat[áa]logo (?:informa|registra|descreve)", re.I)
ABERTURAS_ETIQUETA = ("o objeto é", "trata-se de")
RE_MEDIDA_TXT = re.compile(r"(\d+(?:[.,]\d+)?)\s*cm", re.I)


def tem_atribuicao(texto):
    return re.search(r"(?<![a-zà-ú])(registro|catálogo|catalogo)", texto or "", re.I) is not None


def verificar(item):
    """Devolve os problemas como pares (chave, detalhe). A chave é estável, para somar o
    mesmo problema entre lotes; o detalhe é o que muda de item para item.

    As checagens que dependem de campos que só a v6 produz (o enquadramento decidido na
    observação) são puladas quando o campo não existe — é o que permite medir os lotes
    antigos sem inventar dado que eles não têm. A escala, por ser função só do registro,
    vale para todos."""
    p = []
    alt = item.get("alt_text", "") or ""
    d = item.get("descricao_objeto", "") or ""
    a, dl = alt.lower(), d.lower().strip()
    registro = item.get("registro", {}) or {}

    if item.get("json_valido") is False:
        p.append(("json_invalido", ""))
    if "enquadramento_ok" in item and not item["enquadramento_ok"]:
        p.append(("obs_sem_enquadramento", ""))

    povo = (registro.get("Povo") or "").strip()
    if povo and povo.split()[0].lower() not in a:
        p.append(("povo_ausente_no_alt", povo))
    # nome de povo é nome próprio: "Fuso xavante" está errado (5ª adjudicação)
    if povo:
        alvo_povo = povo.split()[0]
        for texto_orig in (item.get("alt_text") or "", item.get("descricao_objeto") or ""):
            if re.search(rf"(?<![a-zà-úA-ZÀ-Ú]){alvo_povo.lower()}(?![a-zà-ú])", texto_orig):
                p.append(("povo_em_minuscula", alvo_povo.lower()))
                break
    if RE_ARTEFATO.search(a):
        p.append(("artefato_no_alt", RE_ARTEFATO.search(a).group(0)))
    if RE_FUNDO_TXT.search(a):
        p.append(("fundo_no_alt", ""))
    if RE_JARGAO_FOTO.search(a):
        p.append(("jargao_de_foto_no_alt", RE_JARGAO_FOTO.search(a).group(0)))
    if len(alt.split()) > 30:
        p.append(("alt_longo", f"{len(alt.split())} palavras"))
    # a revisao do juiz sobre o v7 achou os tres abaixo passando limpos pelo alt:
    if RE_AUSENCIA.search(a):
        p.append(("ausencia_no_alt", RE_AUSENCIA.search(a).group(0)[:30]))
    if RE_MEDIDA_TXT.search(alt):
        p.append(("medida_no_alt", RE_MEDIDA_TXT.search(alt).group(0)))
    if re.search(r"em escala de|escala\s*:", a + chr(10) + (item.get("descricao_objeto") or "").lower()):
        p.append(("molde_de_escala_no_texto", "a moldura da variável ESCALA vazou"))
    if item.get("enquadramento"):
        alt_detalhe = a.strip().startswith("detalhe")
        if alt_detalhe and item["enquadramento"] != "detalhe":
            p.append(("enquadramento_incoerente", "alt diz Detalhe, observação diz inteiro"))
        if not alt_detalhe and item["enquadramento"] == "detalhe":
            p.append(("enquadramento_incoerente", "observação diz detalhe, alt não marca"))
    return p + _verificar_nivel2(item, d, dl, a, registro)


def _verificar_nivel2(item, d, dl, a, registro):
    p = []
    if d:
        if not tem_atribuicao(d):
            p.append(("nivel2_sem_atribuicao", ""))
        if any(dl.startswith(ab) for ab in ABERTURAS_ETIQUETA):
            p.append(("frase_etiqueta", dl[:18]))
        if "a função é" in dl:
            p.append(("frase_etiqueta", "a função é"))
        if "aquisição em" in dl:
            p.append(("aquisicao_em", ""))
        if RE_ARTEFATO.search(dl):
            p.append(("artefato_no_nivel2", RE_ARTEFATO.search(dl).group(0)))
        if RE_AUSENCIA.search(dl):
            p.append(("afirmacao_de_ausencia", RE_AUSENCIA.search(dl).group(0)))
        if RE_FOTO.search(dl):
            p.append(("foto_no_nivel2", RE_FOTO.search(dl).group(0)))
        if len(d.split()) > 140:
            p.append(("nivel2_longo", f"{len(d.split())} palavras"))
        # colagem do registro bruto: nomes de campo com dois-pontos dentro do texto
        # (o 78838 do v7 colou a ficha inteira — e a palavra "Registro" da colagem
        # ainda comprava a checagem de atribuicao)
        m_colagem = re.search(r"(matéria-prima|técnica de confecção|categoria|dimensões)\s*:", dl)
        if m_colagem:
            p.append(("colagem_do_registro", m_colagem.group(1)))
        # estado da federacao citado sem estar em nenhum campo do registro — o modelo
        # preenche por conhecimento de mundo (Kaxinawa -> Acre) quando o campo esta vazio
        reg_txt = " ".join(str(v) for v in registro.values()).lower()
        for uf in ("acre", "amazonas", "pará", "maranhão", "mato grosso", "tocantins",
                   "pernambuco", "rondônia", "roraima", "amapá", "amazônia"):
            if re.search(rf"(?<![a-zà-ú]){uf}(?![a-zà-ú])", dl) and uf not in reg_txt:
                p.append(("estado_sem_fonte", uf))
        marcas = RE_MARCA_ATRIB.findall(d)
        if len(marcas) > 1:
            p.append(("atribuicao_repetida", f"{len(marcas)} marcas no mesmo texto"))
        if RE_FUNCAO_OBVIA.search(dl):
            p.append(("funcao_obvia", RE_FUNCAO_OBVIA.search(dl).group(0)[:40]))
        # medidas: toda medida escrita tem que estar no registro, e a escala é a maior
        nums_txt = [float(x.replace(",", ".")) for x in RE_MEDIDA_TXT.findall(d)]
        nums_reg = [float(x.replace(",", ".")) for x in
                    re.findall(r"\d+(?:[.,]\d+)?", registro.get("Dimensões", "") or "")]
        for t in nums_txt:
            if nums_reg and not any(abs(r - t) <= 1.0 for r in nums_reg):
                p.append(("medida_fora_do_registro", f"{t:g} cm"))
        # 4a revisao: a contagem do CATALOGO prevalece sempre — o modelo nao e bom nisso
        reg_desc = registro.get("Descrição", "") or ""
        cont_reg, cont_txt = contagens(reg_desc), contagens(a + " " + d)
        for palavra, n_reg in cont_reg.items():
            n_txt = cont_txt.get(palavra)
            if n_txt is not None and n_txt != n_reg:
                p.append(("contagem_diverge_do_registro",
                          f"{palavra}: texto diz {n_txt}, registro diz {n_reg}"))
        escala = item.get("escala") or ""
        if escala and nums_txt:
            m = re.search(r"(\d+(?:[.,]\d+)?)", escala)
            if m:
                esc = float(m.group(1).replace(",", "."))
                if not any(abs(esc - t) <= 1.0 for t in nums_txt):
                    p.append(("escala_errada", f"a maior é {esc:g} cm"))
        if "miniatura" in escala and "miniatura" not in dl:
            p.append(("miniatura_nao_declarada", ""))

    for nome, texto in [("alt", a), ("nível 2", d)]:
        if RE_ANALOGIA.search(texto):
            p.append(("padrao_por_analogia", f"{nome}: {RE_ANALOGIA.search(texto).group(0)[:34]}"))

    for nome, texto in [("alt", a), ("nível 2", dl)]:
        if RE_ESPECULACAO.search(texto):
            p.append(("especulacao", f"{nome}: {RE_ESPECULACAO.search(texto).group(0)}"))
        if RE_VAZIA.search(texto):
            p.append(("frase_vazia", f"{nome}: {RE_VAZIA.search(texto).group(0)}"))

    obs = item.get("observacao") or ""
    if obs and artefatos_da_observacao(obs) and not any(
            f.get("tipo") == "artefato_estudio" for f in (item.get("flags") or [])):
        p.append(("artefato_visto_sem_flag", artefatos_da_observacao(obs)[0]["detalhe"][:40]))

    for f in item.get("flags", []) or []:
        det = (f.get("detalhe") or "").lower()
        if (f.get("tipo") == "artefato_estudio" and RE_FUNDO_TXT.search(det)
                and not RE_ARTEFATO.search(det)):
            p.append(("flag_de_fundo", ""))
        if RE_AUSENCIA.search(det):
            p.append(("flag_de_ausencia", ""))
    return p

for obj in objetos:
    if not obj["resolucao_ok"]:
        # sem resolução: nenhum texto foi gerado — as checagens de texto não se aplicam
        obj["problemas"] = []
        print(f"{obj['id']} {obj['titulo'][:28]:28} ⚑ falta_de_resolucao (sem texto, por política)")
        continue
    obj["problemas"] = verificar(obj)
    detalhe = "; ".join(f"{k}{' (' + v + ')' if v else ''}" for k, v in obj["problemas"])
    print(f"{obj['id']} {obj['titulo'][:28]:28} " + ("✓" if not obj["problemas"] else "⚠ " + detalhe))

abano = next(o for o in objetos if o["id"] == 63283)
checks_abano = {
    "enquadramento 'detalhe'": abano["enquadramento"] == "detalhe",
    "alt abre com 'Detalhe'": abano["alt_text"].strip().lower().startswith("detalhe"),
    "nível 2 com atribuição": tem_atribuicao(abano["descricao_objeto"]),
    "290 cm virou metadado_suspeito": any(f["tipo"] == "metadado_suspeito" for f in abano["flags"]),
}
print("\nCaso-referência Abano (63283): " +
      " | ".join(f"{k} {'✓' if v else '✗'}" for k, v in checks_abano.items()))
tipos = collections.Counter(f["tipo"] for o in objetos for f in o["flags"])
chaves = collections.Counter(k for o in objetos for k, _ in o["problemas"])
print(f"Total: {sum(1 for o in objetos if not o['problemas'])}/{len(objetos)} objetos sem problemas")
print(f"flags por tipo: {dict(tipos)}")
print(f"problemas por checagem: {dict(chaves)}")

In [ ]:
# Etapa 8 — Salvar no Drive (arquivo v8 — os resultados anteriores ficam preservados)
resultado = {
    "notebook": "04_pipeline_completo_v8",
    "modelo": MODELO,
    "embedding": "Qwen/Qwen3-Embedding-0.6B (CPU)",
    "rubrica_versao": rubrica["versao"],
    "rag": "híbrido: diretriz da categoria pelo registro + glossário por similaridade (k=2)",
    "prompt_observacao_v3": PROMPT_OBSERVACAO_V3,
    "prompt_redacao_v12": PROMPT_REDACAO_V12,
    "prompt_contradicao": PROMPT_CONTRADICAO,
    "tetos_categoria": TETOS_CATEGORIA,
    "itens": [
        {"id": o["id"], "titulo": o["titulo"], "registro": o["registro"],
         "observacao": o["observacao"], "enquadramento": o["enquadramento"],
         "enquadramento_ok": o["enquadramento_ok"], "artefatos_obs": o["artefatos_obs"],
         "resolucao": o["resolucao"], "resolucao_ok": o["resolucao_ok"],
         "contagem_registro": o.get("contagem_registro", ""),
         "retry": o.get("retry", False),
         "artefatos_secao": o["artefatos_secao"],
         "marca_atribuicao": o["marca_atribuicao"],
         "flags_cor": o["flags_cor"],
         "escala": o["escala"], "contradicao": o.get("contradicao"),
         "alt_bruto": o.get("alt_bruto", ""), "alt_text": o["alt_text"],
         "descricao_objeto": o["descricao_objeto"],
         "flags": o["flags"], "flags_registro": o["flags_registro"],
         "diretrizes_usadas": o["diretrizes_usadas"],
         "json_valido": o["json_valido"], "problemas": o["problemas"]}
        for o in objetos
    ],
}
destino = f"{PROJETO}/resultados/04_pipeline_completo_v8.json"
os.makedirs(os.path.dirname(destino), exist_ok=True)
with open(destino, "w", encoding="utf-8") as f:
    json.dump(resultado, f, ensure_ascii=False, indent=2)
print(f"salvo no Drive ✓  {destino}")

---

## Fim — o que fazer agora

Avise o Claude que o Notebook 04 **v8** terminou. Ele busca o resultado no Drive e roda as duas réguas — `checar_lote.py` (checagens mecânicas) e `checar_gabarito.py` (reincidência dos defeitos adjudicados) — nos lotes v5, v6, v7 e v8.

**O critério de saída é explícito** (decisão do Eduardo): a etapa só se encerra quando o resultado for satisfatório — **zero reincidência dos defeitos adjudicados** e régua limpa nas checagens mecânicas (fundo, medida/ausência no alt, molde de variável, colagem de registro, escala, contagem, marca de atribuição). O que sobrar deve ser da classe que flags + revisão humana existem para cobrir.

Números novos para olhar: **retries** (quantos textos precisaram da segunda rodada — mede o que o prompt sozinho não segura), **quarentenas** (quantas medidas/cores saíram do texto por flag) e **falta_de_resolucao** (os itens devolvidos ao dataset).